# Example: Stylized Facts of Log Growth Rate Data

Financial returns exhibit universal statistical patterns — called _stylized facts_ — that deviate from the assumptions of classical models. Understanding these facts is essential for building realistic portfolio models and stress tests. In this example, we compute log growth rates from synthetic training data and examine three key properties: fat-tailed distributions, near-zero autocorrelation of returns, and persistent volatility clustering.

> __Learning Objectives:__
>
> * **Log Growth Rate Computation**: Compute the continuously compounded growth rate (CCGR) matrix from synthetic price data and explore its basic statistical properties
> * **Fat Tails and Non-Normality**: Test whether growth rates follow a Normal or Laplace distribution using the Anderson-Darling test, and estimate the tail index via Hill's estimator
> * **Autocorrelation and Volatility Clustering**: Verify that raw returns are approximately uncorrelated (random walk) while absolute returns exhibit persistent positive autocorrelation (volatility clustering)

Let's dive in and compute these stylized facts from our synthetic data!
___

## Setup, Data, and Prerequisites
We begin by loading the `eCornellAIFinance` package and the frozen synthetic training dataset via `Include.jl`.

In [1]:
include("Include.jl");

### Data
We load the frozen 20-year synthetic training dataset generated from the pre-trained JumpHMM portfolio surrogate. This dataset contains daily close prices for 424 tickers plus a synthetic market index, with a curated market path exhibiting realistic drawdowns, jump clusters, and an 8.3% CAGR.

In [2]:
# load the frozen synthetic training dataset -
training_data = MySyntheticTrainingDataSet();
dataset = training_data["dataset"];
list_of_tickers = training_data["tickers"];  # 424 tickers, sorted alphabetically
println("Loaded $(length(list_of_tickers)) tickers × $(training_data["n_days"]) days ($(training_data["n_years"]) years)")

Loaded 424 tickers × 5040 days (20 years)


___
## Task 1: Compute the Log Growth Rate Matrix
We compute the continuously compounded growth rate (CCGR) for every ticker in the dataset. The growth rate of asset $i$ between time $j-1$ and $j$ is:

$$\boxed{g^{(i)}_{j,j-1} = \frac{1}{\Delta t}\ln\frac{S^{(i)}_j}{S^{(i)}_{j-1}}}$$

where $\Delta t = 1/252$ for daily data (one trading day in years).

> __What are we going to do?__ We'll compute the growth rate matrix $\mathbf{X} \in \mathbb{R}^{(T-1) \times N}$ where each row is a time step and each column is a ticker. Then we'll visualize the growth rate time series and its distribution for a selected ticker.
>
> __What should you see?__ The growth rate appears as a stationary random process with non-periodic bursts of extreme volatility followed by calm periods. The distribution has a sharp peak near zero and heavy tails compared to a Normal distribution — this is the "fat tails" stylized fact.

In [3]:
growth_rate_array = let

    # initialize -
    Δt = 1.0 / 252.0;
    N = length(list_of_tickers);
    T_total = training_data["n_days"];
    T = T_total - 1;

    # compute growth rate matrix: (T-1) × N -
    X = zeros(T, N);
    for (j, ticker) ∈ enumerate(list_of_tickers)
        prices = dataset[ticker][:, :close];
        for t ∈ 1:T
            X[t, j] = (1.0 / Δt) * log(prices[t + 1] / prices[t]);
        end
    end

    # verify dimensions -
    @assert size(X) == (T, N) "Growth rate matrix should be $(T) × $(N)"
    println("Growth rate matrix: $(T) days × $(N) tickers")
    println("Mean growth rate (annualized, all tickers): $(round(mean(X) * 100, digits=2))%")

    X  # return
end

Growth rate matrix: 5039 days × 424 tickers
Mean growth rate (annualized, all tickers): 5.25%


5039×424 Matrix{Float64}:
 -0.359938   -3.23993    0.856173  …  -7.86761    -3.48241   -1.49973
  1.65676    -3.17143   -0.203948      4.29999    -2.60838    2.16339
 -3.05945    -7.73713   -3.25104      -3.06266     0.194451  -2.24205
  0.506429   -3.11624    1.57306       8.22606     3.50557   -1.44792
  2.24761    12.7993     1.28786       5.81993     7.51661    3.96994
  0.219386   -6.56985    9.26096   …  -9.21139    -1.80551    9.18743
 -4.01906    -9.7019     8.50129       6.02173     1.75818   -0.845041
 -2.90448    -5.32718   -1.63777       1.60682    -5.26245   -0.580788
 -1.2293    -13.9624    -6.39604       0.983003   -4.01154    0.180908
  0.964264  -13.132      0.245026     -4.08996   -11.7834     1.49532
  ⋮                                ⋱                         
  3.66714     0.891087   6.5953    …   2.30107     0.826239   1.9084
 -2.87375    -5.65241   -2.41459      -9.30906    -8.69876   -7.85422
  1.08478     0.562909  -6.85368      -0.750324  -14.3006     1.69623


### Visualize
Select a ticker to explore. Try different tickers to see how the growth rate behavior varies across assets.

In [ ]:
ticker_to_visualize = "MARKET";  # synthetic market index — direct HMM path with full temporal dynamics
@assert haskey(dataset, ticker_to_visualize) "Ticker $(ticker_to_visualize) not in dataset"

**Growth rate time series and distribution:** The left panel shows the daily growth rate over 20 years. The right panel shows the empirical density compared to fitted Normal, Laplace, and Student-t distributions.

> __What should you see?__ The time series shows bursts of high volatility (large positive and negative spikes clustered together) interspersed with calmer periods — this is volatility clustering. The distribution is sharply peaked near zero with heavy tails. The Student-t distribution (which the HMM uses internally) should provide the best fit, followed by Laplace, with Normal being the worst.

In [ ]:
let
    # compute growth rates for the selected ticker -
    Δt = 1.0 / 252.0;
    prices = dataset[ticker_to_visualize][:, :close];
    X = [(1.0/Δt) * log(prices[t+1]/prices[t]) for t ∈ 1:(length(prices)-1)];

    # left: time series -
    p1 = plot(X, label="", lw=1, color=:red, alpha=0.8, linetype=:steppost,
        xlabel="Trading Day Index (AU)", ylabel="Growth rate μ $(ticker_to_visualize) (1/yr)",
        title="$(ticker_to_visualize) Daily Growth Rate",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);

    # fit distributions -
    d_normal = fit_mle(Normal, X);
    d_laplace = fit_mle(Laplace, X);

    # Student-t: profile MLE over ν (standardize data first) -
    μ_X = mean(X); σ_X = std(X);
    Z = (X .- μ_X) ./ σ_X;
    best_ν = 5.0; best_ll = -Inf;
    for ν_cand ∈ [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0, 15.0, 20.0, 30.0]
        ll = sum(logpdf.(TDist(ν_cand), Z));
        if ll > best_ll
            best_ll = ll; best_ν = ν_cand;
        end
    end
    d_tdist = LocationScale(μ_X, σ_X * sqrt((best_ν - 2)/best_ν), TDist(best_ν));

    # right: density comparison -
    p2 = plot(d_normal, lw=3, color=:deepskyblue1, label="Normal (MLE)",
        xlabel="Growth rate μ $(ticker_to_visualize) (1/yr)", ylabel="Density (AU)",
        title="$(ticker_to_visualize) Distribution",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);
    plot!(p2, d_laplace, lw=3, color=:grey34, label="Laplace (MLE)");
    plot!(p2, d_tdist, lw=3, color=:green4, ls=:dash, label="Student-t (ν=$(Int(best_ν)))");
    density!(p2, X, lw=3, color=:red, label="Observed (daily)");

    plot(p1, p2, layout=(1, 2), size=(1100, 420), margin=5Plots.mm)
end

___
## Task 2: Which Distribution Best Describes Growth Rates?
One of the most robust stylized facts of financial returns is that they are **not normally distributed**. Return distributions have fat tails — the density near zero is greater than Normal, and extreme events occur far more often than a Gaussian model predicts. But which distribution fits best?

> __What are we going to do?__ We use the [Anderson-Darling (AD) test](https://en.wikipedia.org/wiki/Anderson%E2%80%93Darling_test) to compare three candidate distributions — Normal, Laplace, and Student-t — by examining their $A^2$ test statistics. Lower $A^2$ means better fit. With large samples ($T \approx 5000$), the AD test has enormous power and may reject _all_ parametric models — so we focus on **relative fit** (which $A^2$ is smallest) rather than binary pass/fail.
>
> __What should you see?__ The Normal should have the largest $A^2$ (worst fit). Student-t should have the smallest (best fit) since the underlying HMM uses Student-t emissions. Laplace falls between them. We also compute Hill's tail index $\alpha$ to quantify how heavy the tails are.

In [ ]:
let
    Δt = 1.0 / 252.0;
    prices = dataset[ticker_to_visualize][:, :close];
    μ = [(1.0/Δt) * log(prices[t+1]/prices[t]) for t ∈ 1:(length(prices)-1)];
    T = length(μ);

    # fit three distributions -
    d_normal = fit_mle(Normal, μ);
    d_laplace = fit_mle(Laplace, μ);

    # Student-t: profile MLE over ν -
    μ_X = mean(μ); σ_X = std(μ);
    Z = (μ .- μ_X) ./ σ_X;
    best_ν = 5.0; best_ll = -Inf;
    for ν_cand ∈ [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0, 15.0, 20.0, 30.0]
        ll = sum(logpdf.(TDist(ν_cand), Z));
        if ll > best_ll
            best_ll = ll; best_ν = ν_cand;
        end
    end
    d_tdist = LocationScale(μ_X, σ_X * sqrt((best_ν - 2)/best_ν), TDist(best_ν));

    # AD tests -
    ad_normal = OneSampleADTest(μ, d_normal);
    ad_laplace = OneSampleADTest(μ, d_laplace);
    ad_tdist = OneSampleADTest(μ, d_tdist);

    # build results DataFrame sorted by A² (lower = better) -
    results = [(name="Normal", A²=ad_normal.A², pval=pvalue(ad_normal)),
               (name="Laplace", A²=ad_laplace.A², pval=pvalue(ad_laplace)),
               (name="Student-t (ν=$(Int(best_ν)))", A²=ad_tdist.A², pval=pvalue(ad_tdist))];
    sort!(results, by=x -> x.A²);

    df = DataFrame(
        "Distribution" => [r.name for r ∈ results],
        "A² statistic" => [round(r.A², digits=2) for r ∈ results],
        "p-value" => [r.pval < 1e-6 ? "<1e-6" : string(round(r.pval, digits=4)) for r ∈ results],
        "Rank" => ["★ BEST", "2nd", "3rd"]
    );

    println("Anderson-Darling Goodness-of-Fit Comparison for $(ticker_to_visualize) (T = $(T)):")
    pretty_table(df, backend=:text, fit_table_in_display_horizontally=false,
        fit_table_in_display_vertically=false, 
        table_format=TextTableFormat(borders=text_table_borders__compact))
    println("\nNote: With T = $(T) observations, the AD test has very high power.")
    println("Focus on the A² ranking (lower = better fit), not just pass/fail.")
end

**Hill's Tail Index:** The tail index $\alpha$ quantifies how heavy the tails of the distribution are. For a Normal distribution, $\alpha \to \infty$. For financial returns, $\alpha$ is typically between 2 and 5 — meaning extreme events occur _much_ more often than a Gaussian model predicts.

> __Hill's estimator__ works by sorting the positive observations in descending order and measuring how quickly the tail decays. A smaller $\alpha$ means heavier tails (more extreme events).

In [ ]:
let
    Δt = 1.0 / 252.0;
    prices = dataset[ticker_to_visualize][:, :close];
    μ = [(1.0/Δt) * log(prices[t+1]/prices[t]) for t ∈ 1:(length(prices)-1)];

    # Hill's estimator on positive tail -
    pos = filter(>(0), μ);
    n = length(pos);
    z = sort(pos, rev=true);

    # compute: α = 1 / ((1/(n-1)) Σ log(z_i / z_n))
    hill_sum = sum(log(z[i] / z[n]) for i ∈ 1:(n-1));
    α_hill = (n - 1) / hill_sum;

    println("Hill's tail index for $(ticker_to_visualize): α ≈ $(round(α_hill, digits=3))")
    println("  (Normal → ∞, Cauchy → 1, typical equity returns → 2-5)")
end

___
## Task 3: Autocorrelation and Volatility Clustering
Two more stylized facts: (1) raw returns are approximately **uncorrelated** (consistent with the random walk hypothesis), and (2) absolute returns show persistent **positive autocorrelation** — this is volatility clustering, where periods of high volatility tend to follow periods of high volatility.

> __What are we going to do?__ We compute the autocorrelation function (ACF) for both raw growth rates $g_i(t)$ and absolute growth rates $|g_i(t)|$. We compare to the 99% confidence band under the null hypothesis of no autocorrelation.
>
> __What should you see?__ The raw ACF should hover near zero at all lags (random walk). The absolute ACF should be significantly positive for short lags (up to 10–50 days), decaying slowly — this is the volatility clustering signature that motivates models like JumpHMM.

**Autocorrelation of raw returns:** Under the random walk hypothesis, the ACF should be zero for all lags > 0. Significant spikes indicate predictable structure in returns.

> __What should you see?__ Near-zero autocorrelation at all lags, with perhaps a few weak violations in the first few days. This confirms that daily returns are approximately unpredictable — you can't use yesterday's return to forecast today's.

In [ ]:
let
    Δt = 1.0 / 252.0;
    prices = dataset[ticker_to_visualize][:, :close];
    X = [(1.0/Δt) * log(prices[t+1]/prices[t]) for t ∈ 1:(length(prices)-1)];
    T = length(X);
    max_lag = 40;
    lags = collect(0:max_lag);

    acf_raw = autocor(X, lags);
    conf_99 = 2.576 / sqrt(T);
    LINE = conf_99 * ones(length(lags));

    plot(lags, acf_raw, lw=2, color=:red, label="Observed $(ticker_to_visualize)",
        linetype=:steppost,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent,
        xlabel="Lag (trading day)", ylabel="Autocorrelation Growth Rate",
        title="ACF of Raw Returns: $(ticker_to_visualize)", size=(750, 420));
    scatter!(lags, acf_raw, color=:red, msc=:red, ms=3, label="");
    plot!(lags, LINE, lw=2, color=:black, ls=:dash, label="99% confidence");
    plot!(lags, -LINE, lw=2, color=:black, ls=:dash, label="")
end

**Volatility clustering (ACF of |gₜ|):** Volatility clustering means that large moves (positive or negative) tend to cluster together. We measure this by computing the ACF of the _absolute_ growth rate $|g_i(t)|$.

> __What should you see?__ Positive autocorrelation that decays slowly over tens or hundreds of lags — far exceeding the 99% confidence bands. This is one of the strongest and most universal stylized facts. It means: if today was volatile, tomorrow is more likely to be volatile too.

In [ ]:
let
    Δt = 1.0 / 252.0;
    prices = dataset[ticker_to_visualize][:, :close];
    X = [(1.0/Δt) * log(prices[t+1]/prices[t]) for t ∈ 1:(length(prices)-1)];
    T = length(X);
    max_lag = 100;
    lags = collect(0:max_lag);

    acf_abs = autocor(abs.(X), lags);
    conf_99 = 2.576 / sqrt(T);
    LINE = conf_99 * ones(length(lags));

    plot(lags, acf_abs, lw=2, color=:red, label="Observed $(ticker_to_visualize)",
        linetype=:steppost,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent,
        xlabel="Lag (trading day)", ylabel="Autocorrelation |Growth Rate|",
        title="Volatility Clustering: ACF of |gₜ| for $(ticker_to_visualize)", size=(750, 420));
    plot!(lags, LINE, lw=2, color=:black, ls=:dash, label="99% confidence");
    plot!(lags, -LINE, lw=2, color=:black, ls=:dash, label="")
end

___
## Summary

> __Key Takeaways:__
>
> * **Growth rates are not normally distributed**: The Anderson-Darling $A^2$ statistic clearly ranks Normal as the worst fit, with Student-t and Laplace capturing the fat-tailed, peaked shape of empirical returns far better — with large samples, focus on _relative_ fit ($A^2$ ranking) rather than binary reject/fail-to-reject
> * **Raw returns are approximately uncorrelated**: The ACF of $g_i(t)$ drops to near zero at all lags, consistent with the random walk hypothesis — you cannot use past returns to predict future returns
> * **Volatility clusters persistently**: The ACF of $|g_i(t)|$ shows significant positive autocorrelation that decays slowly over tens of days — large moves beget large moves, motivating regime-switching models like JumpHMM that capture this temporal dependence

These stylized facts explain why classical models (which assume i.i.d. Normal returns) fail under stress, and why the JumpHMM surrogate model — which uses Student-t emissions with regime switching and Poisson jumps — produces more realistic synthetic data for portfolio analysis.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.